In [ ]:
import pandas as pd
import os
current_dir = os.getcwd()
df = pd.read_csv(
    "/home/pooja-saxena/PoojaVault/Professional/Workbench/Projects/Employment/Thunen/vacoda/data/raw/Comtrade - 439623.rpt",
    # delim_whitespace=True,
    skiprows=[1],  # only skip dashed line
    decimal=",",
)

df.columns = df.columns.str.strip()

print(df.head())

                                                                                                      CD_ID       CD_HS CD_Reporter CD_Partner CD_Commodity CD_Year CD_Flow CD_Unit CD_Value               CD_ReportedCurrency CD_ConversionFactor CD_Netweight_kg        CD_Sup_Quantity        CD_EstimationCode CD_ReportedClassification CD_HT CD_PartnerSource                                   CD_Valuation CD_FlowsInDataset CD_UnitMult CD_Decimals CD_IDUpload CD_IDReporter CD_IDPartner CD_IDCommodity CD_IDUnit   CD_as_reported CD_Netweight_kg/Value  CD_ActionType
45738088    H0    108         156        940360... 001069            4657                   48    ...                             1401766004415        I                                                                                                                                                                                                                                                                                             

In [5]:
from pathlib import Path
import pandas as pd

# Resolve project root (one level up from notebooks/)
ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path().resolve().parent

rpt_path = ROOT / "data" / "raw" / "Comtrade - 439623.rpt"

df = pd.read_fwf(
    rpt_path,
    skiprows=[1],       # skip the dashed separator line
    na_values=["", " "],
)

print(df.shape)
df.head(3)


(398586, 8)


,CD_ID,CD_HS CD_Reporter CD_Partner CD_Commodity CD_Year CD_Flow CD_Unit CD_Value,CD_ReportedCurrency CD_ConversionFactor CD_Netweight_kg,CD_Sup_Quantity,CD_EstimationCode CD_ReportedClassification CD_HT CD_PartnerSource,CD_Valuation CD_FlowsInDataset CD_UnitMult CD_Decimals CD_IDUpload CD_IDReporter CD_IDPartner CD_IDCommodity CD_IDUnit,CD_as_reported CD_Netweight_kg/Value,CD_ActionType
0,45738088,H0 108 156 940360 2002...,"BIF 0,001069 4657",48.0,0 H0 0 ...,CIF MX 1 1 ...,"1 5,1401766004415",I
1,45739425,H0 108 360 940360 2002...,"BIF 0,001069 25",24.0,0 H0 0 ...,CIF MX 1 1 ...,"1 0,347222222222222",I
2,45737948,H0 108 381 940360 2002...,"BIF 0,001069 202",5.0,0 H0 0 ...,CIF MX 1 1 ...,"1 0,590643274853801",I


In [6]:
import re
from pathlib import Path
import pandas as pd

ROOT = Path().resolve().parent
rpt_path = ROOT / "data" / "raw" / "Comtrade - 439623.rpt"

# Derive exact column positions from the dash-separator line (row 1)
# This handles columns whose values contain spaces (e.g. "Last Known Destination")
with open(rpt_path, encoding="utf-8-sig") as f:
    header_line = f.readline().rstrip("\r\n")
    dashes_line = f.readline().rstrip("\r\n")

segs = list(re.finditer(r"-+", dashes_line))
colspecs = [(m.start(), m.end()) for m in segs]
colnames = [header_line[s:e].strip() for s, e in colspecs]

df = pd.read_fwf(
    rpt_path,
    colspecs=colspecs,
    names=colnames,
    skiprows=2,          # skip header row + dashes row
    encoding="utf-8-sig",
    na_values=["", " "],
    decimal=",",
)

print(df.shape)
df.head(3)


(398586, 29)


,CD_ID,CD_HS,CD_Reporter,CD_Partner,CD_Commodity,CD_Year,CD_Flow,CD_Unit,CD_Value,CD_ReportedCurrency,...,CD_UnitMult,CD_Decimals,CD_IDUpload,CD_IDReporter,CD_IDPartner,CD_IDCommodity,CD_IDUnit,CD_as_reported,CD_Netweight_kg/Value,CD_ActionType
0,45738088,H0,108,156,940360,2002.0,1.0,5.0,906.0,BIF,...,1.0,1.0,70.0,37.0,49.0,6219.0,105.0,1.0,5.140177,I
1,45739425,H0,108,360,940360,2002.0,1.0,5.0,72.0,BIF,...,1.0,1.0,70.0,37.0,113.0,6219.0,105.0,1.0,0.347222,I
2,45737948,H0,108,381,940360,2002.0,1.0,5.0,342.0,BIF,...,1.0,1.0,70.0,37.0,118.0,6219.0,105.0,1.0,0.590643,I


## Merge issue

In [1]:
from vacoda.io.loaders import load_all_interim_parquets
from vacoda.preprocessing.trade import prepare_trade_table
from vacoda.io.loaders import load_raw_sheets
from vacoda.preprocessing.base import standardize_columns  
import pandas as pd

excel_trade = prepare_trade_table(standardize_columns(load_raw_sheets()["main"]))
parquet_trade = prepare_trade_table(load_all_interim_parquets(add_source_col=False))

key_cols = [
    "reporter_code",
    "partner_code",
    "flow_code",
    "unit_code",
    "estimation_code",
]
excel_nulls = excel_trade[excel_trade[key_cols].isna().all(axis=1)]
parquet_nulls = parquet_trade[parquet_trade[key_cols].isna().all(axis=1)]

print(f"Null rows from Excel:   {len(excel_nulls)}")
print(f"Null rows from parquet: {len(parquet_nulls)}")

Null rows from Excel:   0
Null rows from parquet: 8


In [3]:
from vacoda.io.loaders import load_all_interim_parquets
from vacoda.preprocessing.trade import prepare_trade_table
from vacoda.io.loaders import load_raw_sheets
from vacoda.preprocessing.base import standardize_columns
import pandas as pd

excel_trade = prepare_trade_table(standardize_columns(load_raw_sheets()["main"]))
parquet_trade = prepare_trade_table(load_all_interim_parquets(add_source_col=False))

key_cols = [
"reporter_code",
"partner_code",
"flow_code",
"unit_code",
"estimation_code",
]
excel_nulls = excel_trade[excel_trade[key_cols].isna().all(axis=1)]
parquet_nulls = parquet_trade[parquet_trade[key_cols].isna().all(axis=1)]

print(f"Null rows from Excel: {len(excel_nulls)}")
print(f"Null rows from parquet: {len(parquet_nulls)}")

# Load parquets with source_file column to identify which file each null row comes from
parquet_nulls_with_source = load_all_interim_parquets(add_source_col=True)
parquet_trade_src = prepare_trade_table(parquet_nulls_with_source.drop(columns=["source_file"]))

null_mask = parquet_trade_src[key_cols].isna().all(axis=1)
print("Null rows per source file:")
print(parquet_nulls_with_source.loc[null_mask, "source_file"].value_counts())

#Inspect the raw content of those null rows
parquet_nulls_with_source[null_mask]

Null rows from Excel: 0
Null rows from parquet: 8
Null rows per source file:
source_file
Comtrade - 439623    2
Comtrade - 490199    2
Comtrade - 491110    2
Comtrade - 940320    2
Name: count, dtype: int64


,CD_ID,CD_HS,CD_Reporter,CD_Partner,CD_Commodity,CD_Year,CD_Flow,CD_Unit,CD_Value,CD_ReportedCurrency,CD_ConversionFactor,CD_Netweight_kg,CD_Sup_Quantity,CD_EstimationCode,CD_ReportedClassification,CD_HT,CD_PartnerSource,CD_Valuation,CD_FlowsInDataset,CD_UnitMult,CD_Decimals,CD_IDUpload,CD_IDReporter,CD_IDPartner,CD_IDCommodity,CD_IDUnit,CD_as_reported,CD_Netweight_kg/Value,CD_ActionType,source_file
398584,(398584 Zei,en be,roffen),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Comtrade - 439623
398585,Abschlussze,t: 20,6-04-08T14:,0:26.67126,6+02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Comtrade - 439623
838209,(439623 Zei,en be,roffen),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Comtrade - 490199
838210,Abschlussze,t: 20,6-04-08T14:,8:18.76634,5+02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Comtrade - 490199
1229574,(391363 Zei,en be,roffen),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Comtrade - 491110
1229575,Abschlussze,t: 20,6-04-08T14:,2:01.24381,1+02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Comtrade - 491110
1556993,(327417 Zei,en be,roffen),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Comtrade - 940320
1556994,Abschlussze,t: 20,6-04-08T14:,3:58.98014,4+02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Comtrade - 940320
